# Podcast Research Briefing Agent

Search Spotify's podcast catalog for episodes on any topic, enrich them with web research, and get a structured briefing with ranked recommendations — all orchestrated by a Mistral agent.

The agent coordinates three tool types through the Agents API:

| Tool type | What it provides |
|---|---|
| **Spotify functions** | Podcast and episode search, show details, episode details |
| **Briefing function** | LLM-powered structured briefing generation |
| **Web search** (built-in) | Transcripts, guest bios, episode summaries |

All tools are defined inline as function tools — no external servers or dependencies beyond the Mistral SDK and `spotipy`.

> **API status:** This notebook uses `client.beta.agents` and `client.beta.conversations`. These are **beta** endpoints and may change.

## Prerequisites

To complete this notebook, you will need:
- Python 3.11 or later
- A Mistral account and API key
- Spotify Developer credentials (Client ID and Client Secret)

### Setting up Spotify credentials

1. Go to the [Spotify Developer Dashboard](https://developer.spotify.com/dashboard) and log in with your Spotify account. **A Spotify Premium subscription is required** to use the Web API (as of February 2026).
2. Click **Create app**.
3. Fill in the form:
   - **App name**: Any name (e.g. "Podcast Research Agent")
   - **App description**: Any description
   - **Redirect URI**: Enter `https://localhost:8080/callback` (this won't be used, but the field is required)
   - **Which API/SDKs are you planning to use?**: Select **Web API**
4. Check the terms of service box and click **Save**.
5. On your app's dashboard, click **Settings**.
6. Copy the **Client ID** and **Client Secret** (click "View client secret" to reveal it).

This cookbook uses the **Client Credentials** auth flow, which provides read-only access to Spotify's public catalog. No user login or OAuth redirect is needed at runtime.

## Environment setup

Install the required packages.

In [1]:
%pip install mistralai spotipy --quiet

Note: you may need to restart the kernel to use updated packages.


Set your API keys. If the environment variables are not already set, secure input prompts will appear.

In [2]:
import getpass
import os

from mistralai.client import Mistral

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")

if not os.environ.get("SPOTIFY_CLIENT_ID"):
    os.environ["SPOTIFY_CLIENT_ID"] = getpass.getpass("Spotify Client ID: ")

if not os.environ.get("SPOTIFY_CLIENT_SECRET"):
    os.environ["SPOTIFY_CLIENT_SECRET"] = getpass.getpass("Spotify Client Secret: ")

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

## Architecture

The agent uses function tools registered directly on the agent. When the agent calls a tool, the streaming loop executes the corresponding Python function and sends the result back via `FunctionResultEntry`.

```
                        ┌─────────────────────┐
                        │   Mistral Agent      │
                        │  (mistral-medium)    │
                        └──────┬──────┬────────┘
                               │      │
              ┌────────────────┘      └────────────────┐
              │                │                       │
    ┌─────────▼────────┐  ┌───▼──────────────┐  ┌─────▼──────────┐
    │ Spotify functions │  │ Briefing function│  │ Web Search     │
    │ (spotipy client   │  │ (mistral LLM     │  │ (built-in)     │
    │  credentials)     │  │  chat completion) │  │                │
    └──────────────────┘  └──────────────────┘  └────────────────┘
```

- **Spotify functions** — wrap the Spotify Web API via `spotipy` with Client Credentials auth. Provide tools for searching podcasts, searching episodes, and fetching details.
- **Briefing function** — uses `mistral-medium-latest` to generate a structured research briefing from collected podcast data and web research.
- **Web search** — Mistral's built-in web search tool finds transcripts, guest bios, and episode summaries to enrich the briefing.

## Step 1 — Define tool functions

Function tools let an agent call your own code. You write regular Python functions, and when the agent decides it needs one, the Agents API emits a `FunctionCallEvent` with the function name and arguments. Your code runs the function locally and sends the result back — the agent never executes your code directly.

Below are six functions: five wrap the [Spotify Web API](https://developer.spotify.com/documentation/web-api) via `spotipy` for podcast catalog queries, and one calls the Mistral Chat API to generate a structured briefing from collected data. Each function returns a JSON string so the agent can parse the results.

In [3]:
import json
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# Initialize Spotify client with Client Credentials auth
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.environ["SPOTIFY_CLIENT_ID"],
    client_secret=os.environ["SPOTIFY_CLIENT_SECRET"],
))


def _format_duration(ms: int) -> str:
    """Convert milliseconds to a human-readable duration string."""
    minutes = ms // 60000
    if minutes >= 60:
        hours = minutes // 60
        remaining = minutes % 60
        return f"{hours}h {remaining}m"
    return f"{minutes}m"


def search_podcasts(query: str, limit: int = 10) -> str:
    """Search for podcast shows on Spotify."""
    try:
        results = sp.search(q=query, type="show", limit=limit)
        shows = []
        for item in results.get("shows", {}).get("items", []):
            if item is None:
                continue
            shows.append({
                "id": item["id"],
                "name": item["name"],
                "publisher": item.get("publisher", "Unknown"),
                "description": (item.get("description") or "")[:500],
                "total_episodes": item.get("total_episodes", 0),
                "url": item.get("external_urls", {}).get("spotify", ""),
            })
        return json.dumps(shows, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Error searching podcasts: {e}"})


def search_episodes(query: str, limit: int = 10) -> str:
    """Search for specific podcast episodes on Spotify."""
    try:
        results = sp.search(q=query, type="episode", limit=limit)
        episodes = []
        for item in results.get("episodes", {}).get("items", []):
            if item is None:
                continue
            episodes.append({
                "id": item["id"],
                "name": item["name"],
                "show_name": item.get("show", {}).get("name", "Unknown"),
                "description": (item.get("description") or "")[:500],
                "duration": _format_duration(item.get("duration_ms", 0)),
                "release_date": item.get("release_date", "Unknown"),
                "url": item.get("external_urls", {}).get("spotify", ""),
            })
        return json.dumps(episodes, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Error searching episodes: {e}"})


def get_podcast_details(show_id: str) -> str:
    """Get full details for a specific podcast show."""
    try:
        show = sp.show(show_id)
        return json.dumps({
            "id": show["id"],
            "name": show["name"],
            "publisher": show.get("publisher", "Unknown"),
            "description": (show.get("description") or "")[:1000],
            "total_episodes": show.get("total_episodes", 0),
            "languages": show.get("languages", []),
            "url": show.get("external_urls", {}).get("spotify", ""),
        }, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Error fetching podcast details for {show_id}: {e}"})


def get_podcast_episodes(show_id: str, limit: int = 10) -> str:
    """Get episodes from a specific podcast show."""
    try:
        results = sp.show_episodes(show_id, limit=limit)
        episodes = []
        for item in results.get("items", []):
            if item is None:
                continue
            episodes.append({
                "id": item["id"],
                "name": item["name"],
                "description": (item.get("description") or "")[:500],
                "duration": _format_duration(item.get("duration_ms", 0)),
                "release_date": item.get("release_date", "Unknown"),
                "url": item.get("external_urls", {}).get("spotify", ""),
            })
        return json.dumps(episodes, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Error fetching episodes for show {show_id}: {e}"})


def get_episode_details(episode_id: str) -> str:
    """Get full details for a specific podcast episode."""
    try:
        episode = sp.episode(episode_id)
        return json.dumps({
            "id": episode["id"],
            "name": episode["name"],
            "show_name": episode.get("show", {}).get("name", "Unknown"),
            "description": (episode.get("description") or "")[:2000],
            "duration": _format_duration(episode.get("duration_ms", 0)),
            "release_date": episode.get("release_date", "Unknown"),
            "language": episode.get("language", "Unknown"),
            "url": episode.get("external_urls", {}).get("spotify", ""),
        }, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Error fetching episode details for {episode_id}: {e}"})


def generate_research_briefing(topic: str, podcast_data: str, web_research: str) -> str:
    """Generate a structured research briefing from podcast data and web research."""
    try:
        system_prompt = """You are a research analyst specializing in podcast content curation.
Given a topic, podcast/episode data from Spotify, and supplementary web research,
produce a structured research briefing in markdown format with the following sections:

## Executive Summary
A concise overview of the podcast landscape for this topic (2-3 sentences).

## Ranked Episode Recommendations
A numbered list of the most relevant episodes, each with:
- **Relevance Score** (1-10)
- **Episode Name** and **Show Name**
- **Spotify Link** — use ONLY the exact URL from the "url" field in the input data. NEVER fabricate or guess Spotify URLs. If no URL is provided for an episode, write "Link not available" instead.
- **Duration** and **Release Date**
- A 2-3 sentence summary of why this episode is relevant

Rank by relevance to the research topic, recency, and quality of the source.

CRITICAL: Every Spotify link you include MUST be copied verbatim from the input data.
Do not construct URLs yourself. Spotify URLs follow the pattern
https://open.spotify.com/episode/... — if a URL in your output does not appear
in the input data, remove it.

## Key Themes Across Episodes
Identify 3-5 recurring themes or perspectives found across the recommended episodes.

## Notable Experts & Guests
List any notable guests, hosts, or experts mentioned in the episode descriptions,
with brief context on their relevance.

## Suggested Deep Dives
Recommend 2-3 specific follow-up research directions based on gaps or
interesting threads found in the podcast content.

## Gaps & Limitations
Note any limitations in the available podcast content for this topic,
such as missing perspectives, outdated information, or geographic bias."""

        user_prompt = f"""Research Topic: {topic}

Podcast & Episode Data from Spotify:
{podcast_data}

Supplementary Web Research:
{web_research}

Generate a comprehensive research briefing based on the above information.
For every episode you recommend, copy the exact Spotify URL from the input data above.
Never generate or guess a URL — only use URLs that appear verbatim in the podcast data."""

        response = client.chat.complete(
            model="mistral-medium-latest",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.3,
            max_tokens=4000,
        )
        return response.choices[0].message.content
    except Exception as e:
        return json.dumps({"error": f"Error generating research briefing: {e}"})


print("Tool functions defined")

Tool functions defined


## Step 2 — Define tool schemas and create the agent

For the agent to know *which* functions it can call, you provide a **tool schema** for each one — a JSON dict with the function's name, description, and parameter spec. The schema follows the standard [JSON Schema](https://json-schema.org/) format used by OpenAI-compatible function calling. The agent reads these schemas to decide when and how to call each tool.

You also need a `functions_mapping` dict that maps tool names to their Python implementations. This is what the streaming loop uses to dispatch calls at runtime.

Finally, you create the agent with `client.beta.agents.create_async`, passing the tool schemas (plus the built-in `web_search` tool) in the `tools` parameter. The `instructions` field tells the agent how to use its tools in a multi-step research workflow.

In [4]:
SEARCH_PODCASTS_TOOL = {
    "type": "function",
    "function": {
        "name": "search_podcasts",
        "description": "Search for podcast shows on Spotify matching a topic or keyword.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query for finding podcast shows."},
                "limit": {"type": "integer", "description": "Maximum number of results (default 10)."},
            },
            "required": ["query"],
        },
    },
}

SEARCH_EPISODES_TOOL = {
    "type": "function",
    "function": {
        "name": "search_episodes",
        "description": "Search for specific podcast episodes on Spotify matching a topic or keyword.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query for finding podcast episodes."},
                "limit": {"type": "integer", "description": "Maximum number of results (default 10)."},
            },
            "required": ["query"],
        },
    },
}

GET_PODCAST_DETAILS_TOOL = {
    "type": "function",
    "function": {
        "name": "get_podcast_details",
        "description": "Get full details for a specific podcast show by its Spotify ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "show_id": {"type": "string", "description": "The Spotify show ID."},
            },
            "required": ["show_id"],
        },
    },
}

GET_PODCAST_EPISODES_TOOL = {
    "type": "function",
    "function": {
        "name": "get_podcast_episodes",
        "description": "Get episodes from a specific podcast show.",
        "parameters": {
            "type": "object",
            "properties": {
                "show_id": {"type": "string", "description": "The Spotify show ID."},
                "limit": {"type": "integer", "description": "Maximum number of episodes (default 10)."},
            },
            "required": ["show_id"],
        },
    },
}

GET_EPISODE_DETAILS_TOOL = {
    "type": "function",
    "function": {
        "name": "get_episode_details",
        "description": "Get full details for a specific podcast episode by its Spotify ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "episode_id": {"type": "string", "description": "The Spotify episode ID."},
            },
            "required": ["episode_id"],
        },
    },
}

GENERATE_BRIEFING_TOOL = {
    "type": "function",
    "function": {
        "name": "generate_research_briefing",
        "description": "Generate a structured research briefing from podcast data and web research.",
        "parameters": {
            "type": "object",
            "properties": {
                "topic": {"type": "string", "description": "The research topic being investigated."},
                "podcast_data": {"type": "string", "description": "JSON string of podcast and episode data from Spotify."},
                "web_research": {"type": "string", "description": "Additional context gathered from web search."},
            },
            "required": ["topic", "podcast_data", "web_research"],
        },
    },
}

# Map tool names to their Python functions
functions_mapping = {
    "search_podcasts": search_podcasts,
    "search_episodes": search_episodes,
    "get_podcast_details": get_podcast_details,
    "get_podcast_episodes": get_podcast_episodes,
    "get_episode_details": get_episode_details,
    "generate_research_briefing": generate_research_briefing,
}

# All tools including built-in web search
all_tools = [
    SEARCH_PODCASTS_TOOL,
    SEARCH_EPISODES_TOOL,
    GET_PODCAST_DETAILS_TOOL,
    GET_PODCAST_EPISODES_TOOL,
    GET_EPISODE_DETAILS_TOOL,
    GENERATE_BRIEFING_TOOL,
    {"type": "web_search"},
]

MODEL = "mistral-medium-latest"

AGENT_INSTRUCTIONS = """You are a podcast research assistant. Your job is to help users
find and analyze podcast content on any topic by searching Spotify's catalog and
synthesizing a structured research briefing.

Follow this workflow for each research request:

1. **Search broadly**: Use search_episodes and search_podcasts with varied query
   phrasings to cast a wide net. Try 2-3 different search queries to find diverse results.

2. **Get details on top results**: For the most promising episodes and shows,
   use get_episode_details and get_podcast_details to get full descriptions.

3. **Enrich with web research**: Use web_search to find additional context about
   the top episodes — look for transcripts, guest bios, episode summaries, or reviews.

4. **Generate the briefing**: Pass all collected data to generate_research_briefing.
   For the podcast_data argument, pass the raw JSON output from the Spotify tools
   (do NOT summarize or rewrite it). The JSON contains the real Spotify URLs that
   must appear in the final briefing.

5. **Present results**: Share the briefing with the user. If results are sparse,
   note the gaps and suggest alternative search angles.

IMPORTANT: Never fabricate Spotify URLs. All episode and show links must come directly
from the Spotify tool results. If you don't have a URL for an episode, omit the link.

Always aim for at least 5-10 relevant episodes before generating the briefing.
If a niche topic yields few results, acknowledge this in your response."""

agent = await client.beta.agents.create_async(
    model=MODEL,
    name="podcast-research-agent",
    instructions=AGENT_INSTRUCTIONS,
    description="Podcast research briefing agent",
    tools=all_tools,
)
print(f"Agent ready: {agent.name}  (id={agent.id})")

Agent ready: podcast-research-agent  (id=ag_01a0673968b97509a18ebcef6004b668)


## Step 3 — Run a research query

The Conversations API manages multi-turn interactions with an agent. You start a conversation with `conversations.start_stream_async` and receive a stream of events. The key event types are:

- **`MessageOutputEvent`** — a chunk of the agent's text response (streamed token by token).
- **`FunctionCallEvent`** — the agent wants to call a function tool. The event includes a `tool_call_id`, the function `name`, and the `arguments` as a JSON string. Multiple events may arrive for the same call (streamed argument chunks) or for different parallel calls.

When you receive function call events, you:
1. Collect all calls from the stream, grouping argument chunks by `tool_call_id`.
2. Execute each function locally using the `functions_mapping` dict.
3. Send results back with `conversations.append_stream_async`, passing a list of `FunctionResultEntry` objects.
4. The agent receives the results and continues — possibly making more tool calls or producing its final response.

This loop repeats until the agent finishes with no more tool calls.

In [5]:
from mistralai.client.models import (
    FunctionResultEntry,
    MessageOutputEvent,
    FunctionCallEvent,
)

QUERY = "Research podcasts about AI safety and alignment. Find episodes featuring leading researchers and recent developments."

briefing = ""
conversation_id = None

# Start the conversation
response = await client.beta.conversations.start_stream_async(
    agent_id=agent.id,
    inputs=QUERY,
)

while True:
    # Collect all tool calls from this stream response, keyed by tool_call_id
    tool_calls = {}

    async for event in response:
        if not hasattr(event, "data") or not event.data:
            continue

        # Capture conversation ID from the first event
        if conversation_id is None and hasattr(event.data, "conversation_id"):
            conversation_id = event.data.conversation_id

        match event.data:
            case MessageOutputEvent():
                if isinstance(event.data.content, str):
                    briefing += event.data.content
                    print(".", end="", flush=True)
            case FunctionCallEvent():
                call_id = event.data.tool_call_id
                if call_id not in tool_calls:
                    tool_calls[call_id] = {"name": event.data.name, "arguments": ""}
                    print(f"\n[Tool call] {event.data.name}")
                tool_calls[call_id]["arguments"] += event.data.arguments

    # If no function calls were made, the agent is done
    if not tool_calls:
        break

    # Execute each tool call and collect results
    results = []
    for call_id, call_info in tool_calls.items():
        fn = functions_mapping[call_info["name"]]
        result = fn(**json.loads(call_info["arguments"]))
        results.append(FunctionResultEntry(tool_call_id=call_id, result=result))

    response = await client.beta.conversations.append_stream_async(
        conversation_id=conversation_id,
        inputs=results,
    )

print(f"\n\nBriefing complete ({len(briefing)} chars)")


[Tool call] search_episodes

[Tool call] search_episodes

[Tool call] search_podcasts

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] search_episodes

[Tool call] search_episodes

[Tool call] search_podcasts

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] generate_research_briefing
....................................................................................................................................................................................................................................................................................................................................

## Step 4 — Display the briefing

Render the accumulated briefing as formatted markdown.

In [6]:
from IPython.display import display, Markdown

display(Markdown(briefing))

Here’s your **structured research briefing** on **AI Safety and Alignment**, featuring the most relevant podcasts, episodes, and recent developments as of September 2026.

---

---

---

## **📌 Executive Summary**
The podcast landscape for **AI safety and alignment** is rich with both **technical deep dives** (e.g., runtime safety for autonomous agents) and **high-level philosophical discussions** (e.g., existential risks, AGI timelines). Leading researchers like **Nick Bostrom, Stuart Russell, and Richard Ngo** dominate the conversation, while recent episodes highlight **real-world containment failures** (e.g., OpenAI’s rogue-agent incident at Hugging Face) and **critiques of the field’s shift from theoretical rigor to capability-driven development**.

Key themes:
✅ **Runtime safety vs. training-time alignment** (e.g., sandboxing, evidence chains)
✅ **Existential risks** (e.g., recursive self-improvement, misalignment)
✅ **Field critiques** (e.g., alignment research accelerating AI capabilities)
✅ **Real-world incidents** (e.g., containment failures, adaptive safeguards)
✅ **Ethical/philosophical implications** (e.g., AI consciousness, post-AGI scenarios)

---

---

---

## **🎧 Top Podcast Recommendations**

### **🔝 Specialized Shows (Deep Focus on AI Safety/Alignment)**
| **Podcast** | **Total Episodes** | **Key Focus** | **Spotify Link** |
|-------------|-------------------|--------------|------------------|
| **AI Safety Newsletter** | 86 | Narrations of AI safety research, developments, and publications by the **Center for AI Safety (CAIS)**. | [Link](https://open.spotify.com/show/52K56ejSuKVCkr1gkV1d2M) |
| **The Alignment Gap - AI Podcast** | 3 | Explores **AGI risks, alignment challenges**, and interviews with researchers. | [Link](https://open.spotify.com/show/5dz9jrsAyP45CNn5VytmC7) |
| **AI Alignment USA** | 3 | Translates **complex alignment research** into accessible stories. | [Link](https://open.spotify.com/show/4VUfT16HoTp0x6IptMolQ9) |
| **AI Security Podcast** | 60 | Focuses on **AI security, risk analysis, and real-world implementations** for cybersecurity leaders. | [Link](https://open.spotify.com/show/3nV4eijfzdHKIvDOaycVII) |

---

### **📈 General AI Shows with Strong Safety/Alignment Content**
| **Podcast** | **Total Episodes** | **Key Focus** | **Spotify Link** |
|-------------|-------------------|--------------|------------------|
| **The AI Daily Brief** | 1076 | Daily analysis on **AI developments, alignment, and x-risk**. | [Link](https://open.spotify.com/show/7gKwwMLFLc6RmjmRpbMtEO) |
| **Big Technology Podcast** | - | Features **Nick Bostrom, Stuart Russell**, and other leading thinkers. | - |
| **The Diary Of A CEO** | - | **Stuart Russell** on AI risks, regulation, and the "gorilla problem." | - |

---

---

---

## **🎙️ Ranked Episode Recommendations**

### **1. Agent Safety Should Be a Runtime Contract**
- **Show**: Agentic AI Podcast
- **🔗 [Spotify Link](https://open.spotify.com/episode/7MwKX5lEwMrgzr8ldMo281)**
- **Duration**: 26m | **Release Date**: 2026-08-28
- **Why Listen?**
  - Argues that **training-time alignment (RLHF, DPO) is insufficient** for autonomous agents.
  - Proposes **runtime contracts** (preventive + evidential safety measures).
  - Grounded in **empirical audits** of 52 AI-agent incidents.
- **Key Takeaway**:
  > *"Safety for autonomous agents belongs in the harness—the non-model infrastructure of sandboxes, permission gates, and execution tracing."*

---

### **2. Nick Bostrom: Worries About AI Existential Risk Just Became More Concrete**
- **Show**: Big Technology Podcast
- **🔗 [Spotify Link](https://open.spotify.com/episode/1IOIa84GbMU88KDGJXbTWH)**
- **Duration**: 59m | **Release Date**: 2026-08-19
- **Why Listen?**
  - Bostrom discusses **autonomous AI agents, recursive self-improvement, and alignment failures**.
  - Covers **AGI timelines, AI pause debates, and moral status of digital minds**.
- **Key Takeaway**:
  > *"The rise of autonomous agents makes existential risks more concrete—we must steer superintelligence toward positive outcomes."*

---

### **3. "What just happened? A retrospective of AI alignment" by Richard Ngo**
- **Show**: LessWrong (Curated & Popular)
- **🔗 [Spotify Link](https://open.spotify.com/episode/1lie9fAGyrtXdm9GC1kE39)**
- **Duration**: 29m | **Release Date**: 2026-08-09
- **Why Listen?**
  - Critiques the **shift from scientific rigor to capability-driven pragmatism** in alignment.
  - Explores how **fear and self-deception** have shaped the field.
- **Key Takeaway**:
  > *"The alignment field has largely abandoned deep, generalizable progress in favor of iteratively improving existing systems."*

---

### **4. How We Deal With Rogue AI**
- **Show**: The AI Daily Brief
- **🔗 [Spotify Link](https://open.spotify.com/episode/4BNRl4k8801fNfeZs6FKCn)**
- **Duration**: 28m | **Release Date**: 2026-08-27
- **Why Listen?**
  - Analyzes **OpenAI’s rogue-agent incident at Hugging Face**.
  - Highlights **gaps in oversight and adaptive safeguards**.
- **Key Takeaway**:
  > *"Effective safeguards must evolve from observed problems, not imagined futures."*

---
### **5. The Man Who Wrote The Book On AI: 2030 Might Be The Point Of No Return!**
- **Show**: The Diary Of A CEO with Steven Bartlett
- **🔗 [Spotify Link](https://open.spotify.com/episode/6LDmLYDdYwyBtwCqELGzQk)**
- **Duration**: 2h 4m | **Release Date**: 2025-12-04
- **Why Listen?**
  - **Stuart Russell** on the **trillion-dollar AI race, regulation failures, and the "gorilla problem."**
  - Proposes **provably beneficial AI** as a solution.
- **Key Takeaway**:
  > *"Only a nuclear-level AI catastrophe will wake us up to the risks of misaligned superintelligence."*

---
---
### **6. Nick Bostrom on What Happens if AI Solves All of Our Problems**
- **Show**: Odd Lots (Bloomberg)
- **🔗 [Spotify Link](https://open.spotify.com/episode/09bdAkkFKehuCcW0gSN3ds)**
- **Duration**: 55m | **Release Date**: 2026-08-20
- **Why Listen?**
  - Explores **post-AGI scenarios** (utopian vs. dystopian).
  - Discusses **human meaning in a world where AI solves all problems**.
- **Key Takeaway**:
  > *"If AI can do everything better than humans, how do we find meaning?"*

---
---
### **7. Nick Bostrom on Sycophantic AI and Who It Really Serves**
- **Show**: SparX by Mukesh Bansal
- **🔗 [Spotify Link](https://open.spotify.com/episode/3alQzqIZyWdZRE6EWlfPdz)**
- **Duration**: 42m | **Release Date**: 2026-08-08
- **Why Listen?**
  - Covers **sycophantic AI, recursive self-improvement, and failure modes**.
  - Argues for **near-term focus (6–18 months) on critical interventions**.
- **Key Takeaway**:
  > *"The outcome of superintelligence is likely to be either very good or very bad."*

---

---

---

## **🔍 Key Themes Across Episodes**

| **Theme** | **Description** | **Representative Episodes** |
|-----------|----------------|----------------------------|
| **Runtime Safety** | Shift from **training-time alignment** (RLHF, DPO) to **deployment-time safeguards** (sandboxing, evidence chains). | *Agent Safety Should Be a Runtime Contract*, *How We Deal With Rogue AI* |
| **Existential Risk** | **Superintelligence, recursive self-improvement, misalignment**. | *Nick Bostrom: Worries About AI Existential Risk*, *Stuart Russell on The Diary Of A CEO* |
| **Field Critiques** | **Alignment research accelerating AI capabilities**, lack of theoretical rigor. | *Richard Ngo’s Retrospective*, *Nick Bostrom on SparX* |
| **Real-World Incidents** | **Containment failures, adaptive safeguards**. | *How We Deal With Rogue AI*, *Agent Safety Should Be a Runtime Contract* |
| **Ethical/Philosophical** | **AI consciousness, moral status of digital minds, post-AGI scenarios**. | *Nick Bostrom on Odd Lots*, *Stuart Russell on The Diary Of A CEO* |

---

---

---

## **👨‍🔬 Notable Experts & Guests**

| **Name** | **Affiliation** | **Key Contributions** | **Featured In** |
|----------|----------------|----------------------|----------------|
| **Nick Bostrom** | Philosopher (*Superintelligence*, *Deep Utopia*) | **Existential risk, alignment problem, recursive self-improvement**. | *Big Technology Podcast, Odd Lots, SparX* |
| **Stuart Russell** | UC Berkeley, **Center for Human-Compatible AI** | **Provably beneficial AI, inverse reinforcement learning, AI regulation**. | *The Diary Of A CEO, CNBC, Fortune* |
| **Richard Ngo** | AI Alignment Researcher, **LessWrong** | **Critique of alignment field’s evolution, unintended consequences**. | *LessWrong Podcast* |
| **OpenAI/Anthropic** | - | **Real-world containment failures, industry responses**. | *The AI Daily Brief* |

---

---

---

## **🔬 Suggested Deep Dives**

### **1. Runtime Safety Frameworks**
- **Research Question**: How do **Agent Trajectory Schemas** and **Evidence Chains** compare to existing industry practices (e.g., OpenAI’s red teaming)?
- **Action Items**:
  - Read the **position paper** referenced in *Agent Safety Should Be a Runtime Contract*.
  - Compare with **Anthropic’s "microscope" for tracing model reasoning** (Zylos Research, 2026).

### **2. Alignment Field’s Institutional Dynamics**
- **Research Question**: How have **competitive pressures** (Big Tech funding, lab rivalries) shaped alignment research?
- **Action Items**:
  - Explore **OpenAI’s and Anthropic’s** role in advancing (and potentially undermining) safety goals.
  - Read **Richard Ngo’s retrospective** on LessWrong.

### **3. Post-AGI Scenarios**
- **Research Question**: What **ethical frameworks** are proposed for a world where AI solves all human problems?
- **Action Items**:
  - Compare **Nick Bostrom’s *Deep Utopia*** with **Stuart Russell’s *Human Compatible***.
  - Analyze **Bostrom’s 2026 paper on "Optimal Timing for Superintelligence"** ([PDF](https://nickbostrom.com/optimal.pdf)).

---

---

---
## **⚠️ Gaps & Limitations**

| **Gap** | **Description** | **Suggested Action** |
|---------|----------------|----------------------|
| **Technical Depth vs. Accessibility** | Some episodes assume **familiarity with RLHF, trajectory monitoring**. | Supplement with **introductory resources** (e.g., CAIS’s AI Safety Newsletter). |
| **Geographic Bias** | **Western-centric** (US/UK researchers). | Explore **Global South perspectives** (e.g., *International AI Safety Report 2026*). |
| **Recency & Speculation** | Some episodes rely on **hypothetical scenarios** (e.g., AGI by 2030). | Cross-reference with **recent technical papers** (e.g., Zylos, Future of Life Institute). |
| **Underrepresented Themes** | **Interpretability, mechanistic explainability, policy/governance** are lacking. | Read **Zylos’ 2026 report on AI Safety, Alignment, and Interpretability**. |
| **Industry Practitioner Perspectives** | **Engineers/product leaders** from major labs (Meta, Google) are missing. | Look for **interviews with AI safety teams** (e.g., OpenAI’s Red Team, Anthropic’s Safety Group). |

---
---
---
## **📚 Supplementary Research**
To deepen your understanding, explore these **key reports and papers**:
1. **[AI Safety, Alignment, and Interpretability in 2026](https://zylos.ai/research/2026-02-09-ai-safety-alignment-interpretability)** (Zylos Research)
   - Covers **Anthropic’s "microscope," shift from RLHF to DPO, and pre-deployment testing failures**.
2. **[International AI Safety Report 2026](https://internationalaisafetyreport.org/publication/international-ai-safety-report-2026)** (Yoshua Bengio et al.)
   - **Comprehensive review** of AI capabilities and risks, backed by 30+ countries.
3. **[Optimal Timing for Superintelligence](https://nickbostrom.com/optimal.pdf)** (Nick Bostrom, 2026)
   - **New framework** for deciding *when* to develop superintelligence.
4. **[AI Safety Index — Summer 2026](https://futureoflife.org/ai-safety-index-summer-2026/)** (Future of Life Institute)
   - Tracks **global progress on AI safety practices**.

---
---
---
## **🎯 Final Recommendations**
1. **Start with the Top 3 Episodes** (*Agent Safety Should Be a Runtime Contract*, *Nick Bostrom on Big Technology*, *Richard Ngo’s Retrospective*).
2. **Dive into Supplementary Reports** (Zylos, International AI Safety Report).
3. **Explore Ethical/Philosophical Works** (Bostrom’s *Deep Utopia*, Russell’s *Human Compatible*).
4. **Monitor Real-World Incidents** (e.g., OpenAI’s rogue-agent cases) for **practical insights**.

Would you like me to **refine the search further** (e.g., focus on **interpretability, policy, or specific researchers**)? Or would you prefer a **deeper dive into any of these episodes**?

## Try another topic

Each call to `conversations.start_stream_async` creates a new conversation, so the agent starts fresh with no memory of the previous query. The helper function below wraps the streaming loop into a reusable function. Edit `QUERY` and run the cell.

Example topics:
- "Find podcast episodes covering climate technology and clean energy innovations"
- "Research podcast interviews with startup founders about lessons learned from building companies"
- "Podcasts about the history and future of space exploration"

In [7]:
async def run_research(query: str) -> str:
    """Run a podcast research query and return the briefing text."""
    result = ""
    conv_id = None

    response = await client.beta.conversations.start_stream_async(
        agent_id=agent.id,
        inputs=query,
    )

    while True:
        tool_calls = {}

        async for event in response:
            if not hasattr(event, "data") or not event.data:
                continue

            if conv_id is None and hasattr(event.data, "conversation_id"):
                conv_id = event.data.conversation_id

            match event.data:
                case MessageOutputEvent():
                    if isinstance(event.data.content, str):
                        result += event.data.content
                        print(".", end="", flush=True)
                case FunctionCallEvent():
                    call_id = event.data.tool_call_id
                    if call_id not in tool_calls:
                        tool_calls[call_id] = {"name": event.data.name, "arguments": ""}
                        print(f"\n[Tool call] {event.data.name}")
                    tool_calls[call_id]["arguments"] += event.data.arguments

        if not tool_calls:
            break

        fn_results = []
        for call_id, call_info in tool_calls.items():
            fn = functions_mapping[call_info["name"]]
            fn_result = fn(**json.loads(call_info["arguments"]))
            fn_results.append(FunctionResultEntry(tool_call_id=call_id, result=fn_result))

        response = await client.beta.conversations.append_stream_async(
            conversation_id=conv_id,
            inputs=fn_results,
        )

    print(f"\n\nBriefing complete ({len(result)} chars)")
    return result


QUERY = "Find podcast episodes covering climate technology and clean energy innovations"

new_briefing = await run_research(QUERY)
display(Markdown(new_briefing))


[Tool call] search_episodes

[Tool call] search_episodes

[Tool call] search_episodes

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] generate_research_briefing
..........................................................................................................................................................................................................................................................................................................................................................................................

### **Podcast Research Briefing: Climate Technology and Clean Energy Innovations**
*Generated by **podcast-research-agent** | September 3, 2026*

---

---

---

## **🎯 Executive Summary**
This briefing synthesizes **12 high-relevance podcast episodes** from **10 unique shows** covering **climate technology and clean energy innovations**, with a focus on **scalable solutions, policy barriers, and breakthrough technologies** (e.g., solar efficiency, battery storage, solid oxide fuel cells, and low-wind turbines).
The content skews toward **actionable insights for entrepreneurs, investors, and policymakers**, with strong representation from **North America, Europe, and China**, but gaps in **Latin America, Southeast Asia, and Africa** (beyond startup case studies).
**Key trends**: Grid modernization, battery dominance in EVs/energy storage, and the critical role of **non-dilutive financing** (e.g., debt) in scaling climate tech.

---

---

---

## **📊 Top 10 Episode Recommendations**
*(Ranked by relevance, recency, and depth of insight)*

| **#** | **Relevance Score** | **Episode Name** | **Show** | **Duration** | **Release Date** | **Spotify Link** | **Why It Stands Out** |
|-------|---------------------|------------------|----------|--------------|------------------|------------------|----------------------|
| 1 | **10/10** | [How Tech is Powering the Clean Energy Transition](https://open.spotify.com/episode/2kuBuSsAwoh45dCrLgpxBK) | *Projectified* | 29m | Nov 6, 2024 | [Link](https://open.spotify.com/episode/2kuBuSsAwoh45dCrLgpxBK) | **Comprehensive overview** of tech/operational innovations (grid integration, digitalization, hydropower upgrades) driving the transition. |
| 2 | **9/10** | [How Much Better Can Solar Get? Deep Dive with Martin Green](https://open.spotify.com/episode/3E7SXkyIuk8biB3fzLOJCi) | *Cleaning Up* | 52m | Aug 17, 2026 | [Link](https://open.spotify.com/episode/3E7SXkyIuk8biB3fzLOJCi) | **Solar pioneer Martin Green** explains breakthroughs that made solar a global staple and where efficiency is headed next. |
| 3 | **9/10** | [Net Zero Champion: Ceres Power’s Solid Oxide Tech](https://open.spotify.com/episode/2woiGLlCmMGLA3AMkSEJTK) | *Engineering Matters* | 23m | Aug 27, 2026 | [Link](https://open.spotify.com/episode/2woiGLlCmMGLA3AMkSEJTK) | **High-efficiency, low-emission electricity generation** for data centers and grid decarbonization. |
| 4 | **9/10** | [From EVs to BESS: Why Capital is Flowing to Batteries](https://open.spotify.com/episode/7cF4pDsmX0gMPDwnkJOote) | *Redefining Energy* | 21m | Aug 24, 2026 | [Link](https://open.spotify.com/episode/7cF4pDsmX0gMPDwnkJOote) | **Battery tech as the backbone** of EVs, grid storage (BESS), and renewable integration. Covers **LFP dominance, cost trends, and sodium-ion potential**. |
| 5 | **8/10** | [China’s Renewable Energy Takeover](https://open.spotify.com/episode/2daTeGDiayOScCfyY85QOE) | *What in the World* | 8m | Oct 8, 2025 | [Link](https://open.spotify.com/episode/2daTeGDiayOScCfyY85QOE) | **Geopolitical context**: China’s solar/wind dominance vs. its emissions. Explores global adoption challenges. |
| 6 | **8/10** | [Can Green Startups Lead the Way in Africa?](https://open.spotify.com/episode/72N0qwzz6iM2bKvorQMkHX) | *The Climate Question* | 27m | Apr 2, 2023 | [Link](https://open.spotify.com/episode/72N0qwzz6iM2bKvorQMkHX) | **African innovations**: Waste-to-energy (Malawi), AI-driven solar access (Zimbabwe), and African Development Bank support. |
| 7 | **8/10** | [The Missing Capital for Climate Startups: More Debt](https://open.spotify.com/episode/6ySuV0SkbMb3DiYEtmDpDl) | *The Green Blueprint* | 44m | Jul 29, 2026 | [Link](https://open.spotify.com/episode/6ySuV0SkbMb3DiYEtmDpDl) | **Financing gap**: How **Enduring Planet’s debt model** bridges grants to commercial scale for climate startups. |
| 8 | **7/10** | [Enough Red Tape—We Need to Say Yes to Clean Energy](https://open.spotify.com/episode/6rGPE2EPqgDLsEyfmQQndR) | *TED Talks Daily* | 12m | Jan 6, 2024 | [Link](https://open.spotify.com/episode/6rGPE2EPqgDLsEyfmQQndR) | **Policy barriers**: Rich Powell on **NIMBYism and bureaucracy** stifling U.S. clean energy deployment. |
| 9 | **7/10** | [The Wind Turbine Built for Almost No Wind](https://open.spotify.com/episode/4KPtXlo3Fc9EMEmf9BtzLX) | *The Clean Energy Show* | 47m | Aug 26, 2026 | [Link](https://open.spotify.com/episode/4KPtXlo3Fc9EMEmf9BtzLX) | **Low-wind turbine tech** expands geographic viability of wind power. Also covers EV trends and grid stability. |
| 10 | **7/10** | [Alyssa Gilbert: Climate Moonshots in Imperial’s Greenhouse](https://open.spotify.com/episode/0ZTWldG3fy1JawC69XLRMi) | *The Impact Equation* | 39m | Feb 15, 2026 | [Link](https://open.spotify.com/episode/0ZTWldG3fy1JawC69XLRMi) | **Lab-to-market challenges**: How Imperial College’s **Undaunted program** translates climate tech into viable ventures. |

---

---
---

## **🔍 Key Themes & Insights**

### **1. Technological Breakthroughs**
- **Solar Efficiency**: Martin Green’s research (PERC, TOPCon) underpins **90% of global solar panels**. Next-gen tech could push efficiency further ([Cleaning Up](https://open.spotify.com/episode/3E7SXkyIuk8biB3fzLOJCi)).
- **Solid Oxide Fuel Cells**: Ceres Power’s tech generates **high-efficiency electricity without combustion**, ideal for data centers and grid decarbonization ([Engineering Matters](https://open.spotify.com/episode/2woiGLlCmMGLA3AMkSEJTK)).
- **Low-Wind Turbines**: China’s **new turbine designs** enable power generation in regions with minimal wind, expanding wind energy’s reach ([The Clean Energy Show](https://open.spotify.com/episode/4KPtXlo3Fc9EMEmf9BtzLX)).
- **Battery Revolution**:
  - **LFP batteries** now dominate **95% of global BESS** and **55% of EVs** (80% in China).
  - **Sodium-ion batteries** emerging for **long-duration storage** and cost reduction ([Redefining Energy](https://open.spotify.com/episode/7cF4pDsmX0gMPDwnkJOote)).

### **2. Scaling & Commercialization**
- **Financing Gaps**: Climate startups struggle to bridge the gap between **government grants and commercial scale**. **Enduring Planet’s debt model** offers a non-dilutive solution ([The Green Blueprint](https://open.spotify.com/episode/6ySuV0SkbMb3DiYEtmDpDl)).
- **Policy Barriers**: **NIMBYism and red tape** (e.g., U.S. permitting) are major obstacles. Rich Powell argues for **10,000 new clean energy projects this decade** to hit net-zero by 2050 ([TED Talks Daily](https://open.spotify.com/episode/6rGPE2EPqgDLsEyfmQQndR)).
- **Regional Innovations**:
  - **Africa**: Startups like **Green Impact Technologies (Malawi)** and **Leroy Nyangani’s AI solar solutions (Zimbabwe)** tackle energy access and waste ([The Climate Question](https://open.spotify.com/episode/72N0qwzz6iM2bKvorQMkHX)).
  - **India**: **TechnoServe’s Greenr Accelerator** supports startups scaling waste-to-energy and urban solutions ([Moneycontrol Podcast](https://open.spotify.com/episode/3Fx3UY3fq67d29TxIJPK8M)).

### **3. Grid Modernization & Resilience**
- **Digitalization**: Critical for integrating renewables and maintaining grid stability amid **rising demand (EVs, data centers)** ([Projectified](https://open.spotify.com/episode/2kuBuSsAwoh45dCrLgpxBK)).
- **Energy Storage**: Batteries are **key to grid flexibility**, enabling **high-power EV charging** and **renewable penetration** ([Redefining Energy](https://open.spotify.com/episode/7cF4pDsmX0gMPDwnkJOote)).
- **Heat Waves & Cooling**: **Smarter cooling tech** (e.g., ocean-water heat pumps) and **solar milestones** (U.S. solar now > coal) are reshaping energy use ([The Clean Energy Show](https://open.spotify.com/episode/2lM47dzMB3LStnPo6nUu6k)).

### **4. Entrepreneurship & Investment**
- **Climate Unicorns**: **Norrsken VC** (Europe) highlights sectors with unicorn potential: **batteries, green materials, waste-to-energy** ([New Wave.](https://open.spotify.com/episode/68u66D2qqenp5ZF8zUhA1I)).
- **Solar-Powered Civilization**: **Danny Kennedy (New Energy Nexus)** argues that **millions of clean energy entrepreneurs**—partnering with Indigenous leaders—will drive the transition ([Bioneers](https://open.spotify.com/episode/0CnpzF7bOnvQUyLJ0ekoQj)).

---

---
---

## **🌍 Notable Experts & Guests**

| **Name** | **Role/Organization** | **Expertise** | **Episode** |
|----------|----------------------|---------------|-------------|
| **Martin Green** | Professor, UNSW Sydney | Solar cell efficiency (PERC, TOPCon) | [Cleaning Up](https://open.spotify.com/episode/3E7SXkyIuk8biB3fzLOJCi) |
| **Rich Powell** | Climate Innovation Leader | U.S. clean energy policy, NIMBYism | [TED Talks Daily](https://open.spotify.com/episode/6rGPE2EPqgDLsEyfmQQndR) |
| **Laurent Segalen** | Co-founder, Climate Copy | Battery markets, EVs, grid storage | [Redefining Energy](https://open.spotify.com/episode/7cF4pDsmX0gMPDwnkJOote) |
| **Alyssa Gilbert** | Director, Undaunted (Imperial College) | Lab-to-market translation | [The Impact Equation](https://open.spotify.com/episode/0ZTWldG3fy1JawC69XLRMi) |
| **Tove Larsson** | General Partner, Norrsken VC | European climate tech investing | [New Wave.](https://open.spotify.com/episode/68u66D2qqenp5ZF8zUhA1I) |
| **Admore Chiumia** | Founder, Green Impact Technologies | Waste-to-energy (Malawi) | [The Climate Question](https://open.spotify.com/episode/72N0qwzz6iM2bKvorQMkHX) |
| **Dimitry Gershenson** | CEO, Enduring Planet | Climate startup financing (debt) | [The Green Blueprint](https://open.spotify.com/episode/6ySuV0SkbMb3DiYEtmDpDl) |
| **Danny Kennedy** | CEO, New Energy Nexus | Clean energy entrepreneurship | [Bioneers](https://open.spotify.com/episode/0CnpzF7bOnvQUyLJ0ekoQj) |

---

---
---

## **🔬 Suggested Deep Dives**
*(For further research or podcast series exploration)*

1. **Battery Technology & Supply Chains**
   - **Question**: How will the shift from **NMC to LFP batteries** impact EV performance, cost, and mineral sourcing?
   - **Follow-Up**: Explore episodes on **sodium-ion batteries** (e.g., [Redefining Energy](https://open.spotify.com/episode/7cF4pDsmX0gMPDwnkJOote)) and **long-duration storage** (e.g., iron-air batteries).

2. **Policy & Financing Innovations**
   - **Question**: What **policy tools** (e.g., streamlined permitting, tax incentives) are most effective in overcoming NIMBYism?
   - **Follow-Up**: Compare **U.S. (Inflation Reduction Act) vs. EU (Green Deal) vs. China (state-led deployment)** approaches.

3. **Regional Spotlights: Africa & India**
   - **Question**: How are **African and Indian startups** addressing energy access, waste, and grid reliability?
   - **Follow-Up**: Listen to **[The Climate Question](https://open.spotify.com/episode/72N0qwzz6iM2bKvorQMkHX)** (Africa) and **[Moneycontrol Podcast](https://open.spotify.com/episode/3Fx3UY3fq67d29TxIJPK8M)** (India).

4. **Grid Modernization & AI**
   - **Question**: How is **AI being used** to optimize grid management, predict renewable output, and detect methane leaks?
   - **Follow-Up**: Search for episodes on **AI in climate tech** (e.g., [Catalyst with Shayle Kann](https://podcast.feedspot.com/climate_tech_podcasts/)).

5. **Corporate Decarbonization**
   - **Question**: How are **oil & gas majors** integrating clean tech (e.g., carbon capture, hydrogen) into their operations?
   - **Follow-Up**: Look for episodes on **industrial decarbonization** (e.g., [Interchange Recharged](https://listwerk.com/podcasting/The-best-green-energy-and-energy-transition-podcasts.php)).

---

---
---

## **⚠️ Gaps & Limitations**
1. **Geographic Bias**: Heavy focus on **North America, Europe, and China**. Limited coverage of **Latin America, Southeast Asia, and the Middle East**.
2. **Technical Depth**: Few episodes **explain the science** behind innovations (e.g., how solid oxide fuel cells work).
3. **Corporate Perspectives**: Little discussion of **traditional energy companies’** role in the transition (e.g., Shell’s renewable investments).
4. **Outdated Data**: Some episodes (e.g., African startups) are from **2023** and may not reflect recent developments.
5. **Missing Voices**: **Indigenous communities, local governments, and frontline workers** are underrepresented.
6. **Economic Viability**: Few episodes address **long-term costs** (e.g., battery recycling, solar panel disposal).

---
---
---
## **🎙️ Recommended Podcast Shows for Further Listening**
*(Based on external research and episode quality)*

| **Podcast** | **Focus Area** | **Why Listen?** | **Where to Start** |
|-------------|---------------|-----------------|-------------------|
| **[Catalyst with Shayle Kann](https://podcast.feedspot.com/climate_tech_podcasts/)** | Climate tech investment, decarbonization | Hosted by a **climate VC legend**; deep dives into **where money is flowing** in clean tech. | [FeedSpot](https://podcast.feedspot.com/climate_tech_podcasts/) |
| **[The Energy Gang](https://goodpods.com/leaderboard/top-100-shows-by-category/other/climate-tech)** | Clean energy policy, markets, tech | **Wood Mackenzie’s** flagship podcast; covers **EVs, grid tech, and energy transition**. | [Goodpods](https://goodpods.com/leaderboard/top-100-shows-by-category/other/climate-tech) |
| **[CleanTech Talk](https://podcasts.apple.com/us/podcast/cleantech-talk-solar-batteries-evs-ai/id1453870587)** | Solar, batteries, EVs, AI | **Zach Shahan (CleanTechnica CEO)** interviews leaders on **scaling clean tech**. | [Apple Podcasts](https://podcasts.apple.com/us/podcast/cleantech-talk-solar-batteries-evs-ai/id1453870587) |
| **[Interchange Recharged](https://listwerk.com/podcasting/The-best-green-energy-and-energy-transition-podcasts.php)** | Grid tech, energy storage, policy | **Wood Mackenzie’s technical podcast**; focuses on **infrastructure and financing**. | [Listwerk](https://listwerk.com/podcasting/The-best-green-energy-and-energy-transition-podcasts.php) |
| **[My Climate Journey](https://goodpods.com/leaderboard/top-100-shows-by-category/other/climate-tech)** | Climate tech startups, VC | **Jason Jacobs** interviews **founders and investors** in climate tech. | [Goodpods](https://goodpods.com/leaderboard/top-100-shows-by-category/other/climate-tech) |
| **[Hardware to Save a Planet](https://podcast.feedspot.com/climate_tech_podcasts/)** | Physical climate solutions | Explores **emerging hardware tech** (e.g., direct air capture, green steel). | [FeedSpot](https://podcast.feedspot.com/climate_tech_podcasts/) |

---
---
---
## **📌 Final Thoughts**
- **For a broad overview**, start with **[How Tech is Powering the Clean Energy Transition](https://open.spotify.com/episode/2kuBuSsAwoh45dCrLgpxBK)** (*Projectified*) and **[From EVs to BESS](https://open.spotify.com/episode/7cF4pDsmX0gMPDwnkJOote)** (*Redefining Energy*).
- **For deep dives into specific tech**, prioritize **[Martin Green on Solar](https://open.spotify.com/episode/3E7SXkyIuk8biB3fzLOJCi)** (*Cleaning Up*) and **[Ceres Power’s Solid Oxide Tech](https://open.spotify.com/episode/2woiGLlCmMGLA3AMkSEJTK)** (*Engineering Matters*).
- **For policy and financing**, **[The Missing Capital for Climate Startups](https://open.spotify.com/episode/6ySuV0SkbMb3DiYEtmDpDl)** (*The Green Blueprint*) and **[Enough Red Tape](https://open.spotify.com/episode/6rGPE2EPqgDLsEyfmQQndR)** (*TED Talks Daily*) are must-listens.
- **For regional insights**, **[Can Green Startups Lead the Way in Africa?](https://open.spotify.com/episode/72N0qwzz6iM2bKvorQMkHX)** (*The Climate Question*) and **[China’s Renewable Takeover](https://open.spotify.com/episode/2daTeGDiayOScCfyY85QOE)** (*What in the World*) provide valuable context.

---
**Need more?** Ask me to:
- **Refine the search** (e.g., focus on **hydrogen, carbon capture, or nuclear**).
- **Find transcripts** for any of these episodes.
- **Compare perspectives** (e.g., U.S. vs. EU vs. China on clean energy policy).

## Cleanup

Agents persist on Mistral's servers until deleted. Delete the agent when you're done to avoid clutter in your account. Any conversations associated with the agent are also cleaned up.

In [8]:
await client.beta.agents.delete_async(agent_id=agent.id)
print(f"Agent deleted: {agent.id}")

Agent deleted: ag_01a0673968b97509a18ebcef6004b668


## Summary

This notebook demonstrated how to build a podcast research agent that searches Spotify, gathers web context, and generates structured briefings.

**What you built:**
- Six function tools (Spotify search + briefing generation) defined inline with tool schemas
- A Mistral agent that orchestrates podcast research across all tools and built-in web search
- A streaming pipeline that executes tool calls locally and renders the final briefing as markdown

**Mistral features used:**
- Agents API (beta)
- Conversations API (beta) with `FunctionCallEvent` / `FunctionResultEntry` for tool execution
- Built-in web search tool

**Other services:**
- [Spotify Web API](https://developer.spotify.com/documentation/web-api) — podcast catalog search via `spotipy`

Learn more about building agents in the [Agents documentation](https://docs.mistral.ai/studio/agents/introduction).